# Referência: agentes com LangChain, do básico ao loop

**Material de apoio. Não é exercício avaliado.**

No Encontro 1 vocês escreveram tudo à mão: a chamada ao modelo, as ferramentas, o loop do agente. Este notebook mostra **os mesmos conceitos em LangChain**, na mesma ordem, usando o mesmo caso da Aurora Tecnologia.

A ideia não é substituir o que vocês fizeram. É ter a tradução pronta para quando o framework entrar na disciplina (Encontro 4), e uma referência de sintaxe para consultar.

| Seção | Conceito | No Encontro 1 |
|---|---|---|
| 1 | Chamar o modelo | `comum/modelo.py` |
| 2 | Contar tokens e custo | `comum/medidor.py` |
| 3 | Saída estruturada | `comum/resultado.py` |
| 4 | Definir ferramentas | `FERRAMENTAS` em `arquitetura_c.py` |
| 5 | Uma volta de tool calling | o corpo do loop (TODO 4) |
| 6 | O loop completo, à mão | `arquitetura_c.py` |
| 7 | O mesmo agente com `create_agent` | — |
| 8 | Um workflow em LangChain | `arquitetura_b.py` |

## 0. Preparação

Rode uma vez. Usa o mesmo `.env` da raiz do repositório.

In [ ]:
%pip install -q "langchain>=1.0,<2" "langchain-anthropic>=1.0,<2" python-dotenv

In [ ]:
import os
import sys
from pathlib import Path

# Este notebook fica em referencia/. A raiz do repositório é a pasta de cima.
RAIZ = Path.cwd().parent if Path.cwd().name == "referencia" else Path.cwd()
sys.path.insert(0, str(RAIZ))
sys.path.insert(0, str(RAIZ / "encontro-01"))

from dotenv import load_dotenv
load_dotenv(RAIZ / ".env")

MODELO = os.getenv("MODELO", "claude-haiku-4-5-20251001")
print("Modelo:", MODELO)

## 1. Chamar o modelo

`init_chat_model` cria um cliente para qualquer provedor suportado. Trocar de provedor é trocar o texto do modelo, e o resto do código continua igual. É a mesma ideia do nosso `comum/modelo.py`.

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(MODELO, model_provider="anthropic")

resposta = llm.invoke("Em uma frase: o que é um agente de IA?")
print(resposta.content)

Para dar papéis à conversa, passe uma lista de mensagens. A mensagem de sistema é onde entram as regras, como o nosso `REGRAS`.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

mensagens = [
    SystemMessage("Você responde dúvidas de colaboradores da Aurora Tecnologia, em uma frase."),
    HumanMessage("Quantos dias de férias eu tenho por ano?"),
]
resposta = llm.invoke(mensagens)
print(resposta.content)

## 2. Contar tokens e custo

Toda resposta traz `usage_metadata`, com os tokens de entrada e de saída. Com o preço do `.env`, calculamos o custo da chamada, como faz o `comum/medidor.py`.

In [ ]:
from comum.medidor import PRECO_ENTRADA, PRECO_SAIDA

uso = resposta.usage_metadata
custo = (uso["input_tokens"] * PRECO_ENTRADA + uso["output_tokens"] * PRECO_SAIDA) / 1_000_000
print(f"entrada: {uso['input_tokens']} tokens · saída: {uso['output_tokens']} tokens · US$ {custo:.6f}")

## 3. Saída estruturada

No Encontro 1, pedimos JSON no prompt e lemos o texto com `Resultado.de_json`, torcendo para o modelo respeitar o formato. Com `with_structured_output`, o formato vira uma classe Python e o LangChain garante que a resposta chega nela.

É o mesmo contrato do Encontro 1: desfecho, fontes e resposta.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class RespostaAurora(BaseModel):
    """Resposta a uma dúvida de colaborador."""
    desfecho: Literal["responder", "escalar", "nao_sei"]
    fontes: list[str] = Field(description="ids dos documentos que sustentam a resposta")
    resposta: str = Field(description="texto curto para o colaborador")

llm_estruturado = llm.with_structured_output(RespostaAurora)

saida = llm_estruturado.invoke([
    SystemMessage("Responda com base apenas no documento abaixo.\n\n"
                  '<documento id="rh-ferias">São 30 dias de férias a cada 12 meses.</documento>'),
    HumanMessage("Quantos dias de férias eu tenho?"),
])
saida

`saida` é um objeto Python, não texto: dá para usar `saida.desfecho` e `saida.fontes` direto.

## 4. Definir ferramentas

Em LangChain, uma ferramenta é uma função Python com o decorador `@tool`. O que o modelo lê é gerado a partir da função:

- o **nome** da função vira o nome da ferramenta;
- a **docstring** vira a descrição;
- os **tipos dos parâmetros** viram o schema. `Literal` vira uma lista de valores permitidos.

Ou seja, a docstring é o contrato que o modelo lê. Esse é o assunto do Encontro 2.

As duas ferramentas abaixo usam a mesma busca do Encontro 1 (`encontro-01/ferramentas.py`).

In [ ]:
from langchain_core.tools import tool
from ferramentas import buscar_formatado

@tool
def buscar(tema: Literal["rh", "ti", "beneficios", "todos"], consulta: str) -> str:
    """Busca documentos de política interna da Aurora Tecnologia por palavras-chave.
    Devolve até 3 documentos, com id, dono, data de atualização e texto.
    Use o tema 'todos' quando não souber a área ou quando a pergunta envolver mais de uma."""
    return buscar_formatado(tema, consulta)

@tool
def escalar_para_rh(motivo: str) -> str:
    """Encaminha a pergunta para análise humana do RH.
    Use quando a pessoa pergunta se ela tem direito a um benefício."""
    return f"Encaminhado ao RH. Motivo: {motivo}"

print("nome:", buscar.name)
print("descrição:", buscar.description)
print("parâmetros:", buscar.args)

Uma ferramenta também pode ser chamada direto, sem modelo nenhum. É assim que se testa:

In [ ]:
print(buscar.invoke({"tema": "ti", "consulta": "senha bloqueada"})[:400])

## 5. Uma volta de tool calling

`bind_tools` entrega as ferramentas ao modelo. O modelo **não executa** nada: ele devolve um pedido (`tool_calls`), dizendo qual ferramenta quer e com quais argumentos. Quem executa é o nosso código.

In [ ]:
llm_com_ferramentas = llm.bind_tools([buscar, escalar_para_rh])

conversa = [HumanMessage("Minha conta bloqueou depois de errar a senha. O que eu faço?")]
passo = llm_com_ferramentas.invoke(conversa)

print("texto:", passo.content)
print("pedidos de ferramenta:", passo.tool_calls)

Agora executamos o pedido e devolvemos o resultado ao modelo como `ToolMessage`, ligada ao pedido pelo `id`. Com o resultado em mãos, o modelo responde.

In [ ]:
from langchain_core.messages import ToolMessage

FERRAMENTAS_POR_NOME = {"buscar": buscar, "escalar_para_rh": escalar_para_rh}

conversa.append(passo)                     # o pedido do modelo entra no histórico
for pedido in passo.tool_calls:
    ferramenta = FERRAMENTAS_POR_NOME[pedido["name"]]
    resultado = ferramenta.invoke(pedido["args"])
    conversa.append(ToolMessage(content=resultado, tool_call_id=pedido["id"]))

final = llm_com_ferramentas.invoke(conversa)
print(final.content)

Isso foi **uma volta** do loop: o modelo pede, o código executa, o modelo lê o resultado. Um agente é essa volta repetida até um critério de parada.

## 6. O loop completo, à mão

O mesmo loop da `arquitetura_c.py`, agora com objetos do LangChain. Repare que o critério de parada (`MAX_PASSOS`) está no nosso código.

In [ ]:
REGRAS = (
    "Você responde dúvidas internas de colaboradores da Aurora Tecnologia. "
    "Use somente os documentos encontrados pela ferramenta buscar e cite os ids. "
    "Se documentos se contradizem, diga que não sabe com segurança e cite os dois. "
    "Se a pessoa pergunta se ela tem direito a um benefício, chame escalar_para_rh."
)
MAX_PASSOS = 6

def agente_a_mao(pergunta: str) -> str:
    conversa = [SystemMessage(REGRAS), HumanMessage(pergunta)]
    for numero in range(1, MAX_PASSOS + 1):
        passo = llm_com_ferramentas.invoke(conversa)
        conversa.append(passo)
        if not passo.tool_calls:                      # sem pedidos: é a resposta final
            print(f"[{numero} passo(s)]")
            return passo.content
        for pedido in passo.tool_calls:
            print(f"  passo {numero}: {pedido['name']}({pedido['args']})")
            resultado = FERRAMENTAS_POR_NOME[pedido["name"]].invoke(pedido["args"])
            conversa.append(ToolMessage(content=resultado, tool_call_id=pedido["id"]))
    return "Não consegui concluir. Encaminhado ao RH."    # critério de parada atingido

print(agente_a_mao("Vou sair de licença-paternidade. Quantos dias tenho e o que faço para incluir o bebê no plano de saúde?"))

## 7. O mesmo agente com `create_agent`

O LangChain já traz esse loop pronto. As mesmas ferramentas, as mesmas regras e o formato de saída da seção 3, em poucas linhas:

In [ ]:
from langchain.agents import create_agent

agente = create_agent(
    model=llm,
    tools=[buscar, escalar_para_rh],
    system_prompt=REGRAS,
    response_format=RespostaAurora,   # opcional: devolve a resposta no contrato da seção 3
)

resultado = agente.invoke(
    {"messages": [{"role": "user", "content": "Qual é o valor do vale-refeição por dia?"}]},
    config={"recursion_limit": 12},   # o critério de parada do framework
)
resultado["structured_response"]

O `resultado["messages"]` guarda o histórico inteiro. Imprimir esse histórico é a forma mais simples de ver o que o agente decidiu a cada passo, uma prévia do *trace* do Encontro 5.

In [ ]:
for mensagem in resultado["messages"]:
    tipo = type(mensagem).__name__
    if getattr(mensagem, "tool_calls", None):
        for pedido in mensagem.tool_calls:
            print(f"{tipo:<14} pede {pedido['name']}({pedido['args']})")
    else:
        print(f"{tipo:<14} {str(mensagem.content)[:100]}")

**O que se ganha e o que se perde com o loop pronto:**

- **Ganha:** menos código, formato de saída garantido, e o mesmo padrão em todo projeto.
- **Perde:** o loop fica escondido. O critério de parada vira `recursion_limit`, que conta passos internos do framework (cada volta de ferramenta consome mais de um) e, quando estoura, lança um erro em vez de escalar para o RH.

Saber o que o framework faz por baixo é o que permite decidir se ele serve. Por isso o Encontro 1 escreve o loop à mão.

## 8. Um workflow em LangChain

LangChain não serve só para agentes. O workflow da `arquitetura_b.py` fica assim: o modelo classifica com saída estruturada, e o **código** decide o caminho com um `if`. Não há agente e não há loop.

In [ ]:
from ferramentas import buscar as buscar_documentos, formatar

class Classificacao(BaseModel):
    categoria: Literal["rh", "ti", "beneficios", "elegibilidade"]

classificador = llm.with_structured_output(Classificacao)
redator = llm.with_structured_output(RespostaAurora)

def workflow(pergunta: str) -> RespostaAurora:
    categoria = classificador.invoke([
        SystemMessage("Classifique a pergunta do colaborador. "
                      "'elegibilidade' é quando a pessoa pergunta se ELA tem direito a um benefício."),
        HumanMessage(pergunta),
    ]).categoria

    if categoria == "elegibilidade":                   # o código decide, não o modelo
        return RespostaAurora(desfecho="escalar", fontes=[], resposta="Encaminhado ao RH.")

    documentos = "\n\n".join(formatar(d) for d in buscar_documentos(categoria, pergunta))
    return redator.invoke([
        SystemMessage(REGRAS + "\n\nDocumentos:\n\n" + documentos),
        HumanMessage(pergunta),
    ])

workflow("Meu notebook quebrou em casa. O seguro cobre?")

## Resumo: a tradução

| Conceito | Escrito à mão (Encontro 1) | Em LangChain |
|---|---|---|
| Chamar o modelo | `modelo.chamar(...)` | `init_chat_model(...).invoke(...)` |
| Tokens da chamada | `medidor.atual()` | `resposta.usage_metadata` |
| Formato da resposta | JSON no prompt + `Resultado.de_json` | `with_structured_output(Classe)` |
| Ferramenta | dict com nome, descrição e schema | função com `@tool` e docstring |
| Pedido de ferramenta | `resposta.chamadas` | `resposta.tool_calls` |
| Resultado da ferramenta | `modelo.mensagem_de_resultados(...)` | `ToolMessage(...)` |
| Loop do agente | `while not deve_parar(...)` | `create_agent(...)` |
| Critério de parada | `MAX_PASSOS` | `recursion_limit` |

**Próximo passo:** no Encontro 4 entra o LangGraph, a camada por baixo do `create_agent`. É com ele que se monta um roteador com especialistas, com estado que sobrevive a falhas (*checkpoint*).